In [0]:
import os
import re
import pandas as pd

CATALOG = 'workspace'
SCHEMA  = 'default'
VOLUME  = 'bharat_bricks_hacks'
VOL_PATH = f'/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}'
PDF_DIR  = f'{VOL_PATH}/pdfs'

def clean_cols(df):
    df.columns = [
        re.sub(r'[\s,;{}()\n\t=]+', '_', str(c)).strip('_')
        for c in df.columns
    ]
    return df

def save_table(df, table, mode='overwrite'):
    df = clean_cols(df)
    sdf = spark.createDataFrame(df.astype(str))
    sdf.write.format('delta').mode(mode) \
       .option('overwriteSchema', 'true' if mode == 'overwrite' else 'false') \
       .saveAsTable(f'{CATALOG}.{SCHEMA}.{table}')
    print(f'  💾 {CATALOG}.{SCHEMA}.{table}  ({df.shape[0]} rows × {df.shape[1]} cols) [{mode}]')

print(f'✅ Config: {CATALOG}.{SCHEMA}')
print(f'   Volume: {VOL_PATH}')

In [0]:
!pip install PyMuPDF

In [0]:
import fitz  # PyMuPDF

constitution_chunks = []

# Try PDF first
CONSTITUTION_PDF = f'{PDF_DIR}/constitution_of_india.pdf'

if os.path.exists(CONSTITUTION_PDF):
    print(f'📄 Parsing Constitution PDF: {CONSTITUTION_PDF}')
    print(f'   Size: {os.path.getsize(CONSTITUTION_PDF):,} bytes')
    
    doc = fitz.open(CONSTITUTION_PDF)
    full_text = ""
    for page in doc:
        full_text += page.get_text() + "\n"
    page_count = doc.page_count
    doc.close()
    print(f'   Extracted {len(full_text):,} characters from {page_count} pages')
    
    # Parse articles: "Article NNN. Title — ..." or "NNN. Title.—"
    article_pattern = re.compile(
        r'(?:Article\s+)?(\d{1,3}[A-Z]?)\.\s*(.+?)(?:\.\s*[-—–]|\s*[-—–])\s*(.*?)(?=(?:Article\s+)?\d{1,3}[A-Z]?\.\s|\Z)',
        re.DOTALL
    )
    
    # Simpler fallback: split on "Article N." boundaries
    article_splits = re.split(r'(?=\b(?:Article\s+)(\d{1,3}[A-Z]?)\.)', full_text)
    
    # Try to extract articles with a broader pattern
    articles_found = {}
    for m in re.finditer(r'(?:Article\s+)?(\d{1,3}[A-Z]?)\.\s+([^\n]+)', full_text):
        num = m.group(1).strip()
        title = m.group(2).strip()[:200]
        # Get body text (next ~2000 chars after the title)
        start = m.end()
        # Find next article
        next_match = re.search(r'(?:Article\s+)?\d{1,3}[A-Z]?\.\s+', full_text[start:start+3000])
        end = start + (next_match.start() if next_match else 2000)
        body = full_text[start:end].strip()
        body = re.sub(r'\s+', ' ', body)[:2000]
        
        if num not in articles_found and len(body) > 20:
            articles_found[num] = {
                'title': title,
                'body': body,
            }
    
    for num, data in sorted(articles_found.items(), key=lambda x: (int(re.sub(r'[A-Z]', '', x[0]) or '0'), x[0])):
        constitution_chunks.append({
            'chunk_id': f'CONST_A{num}',
            'source': 'Constitution_of_India',
            'doc_type': 'constitutional_law',
            'title': f'Article {num}: {data["title"]}',
            'text': f'Article {num} of the Constitution of India — {data["title"]}. {data["body"]}',
        })
    
    print(f'  ✅ Extracted {len(constitution_chunks)} articles from PDF')
else:
    print(f'⚠️  Constitution PDF not found at {CONSTITUTION_PDF}')
    print(f'   Upload to: Catalog → Volumes → {VOLUME} → pdfs/')

# Fallback: key constitutional articles as curated text (always add these for reliability)
KEY_ARTICLES = [
    ("14", "Equality before law", "constitutional_law",
     "Article 14 — The State shall not deny to any person equality before the law or the equal protection of the laws within the territory of India. This is a fundamental right guaranteeing that no person shall be treated unequally before law. Any arbitrary classification or discrimination by the State violates Article 14."),
    ("19", "Protection of certain rights regarding freedom of speech, etc.", "constitutional_law",
     "Article 19 — (1) All citizens shall have the right to (a) freedom of speech and expression; (b) assemble peaceably and without arms; (c) form associations or unions; (d) move freely throughout the territory of India; (e) reside and settle in any part of the territory of India; (g) practise any profession, or to carry on any occupation, trade or business. These rights are subject to reasonable restrictions under Article 19(2)-(6) in the interests of sovereignty, integrity, security of the State, public order, decency, morality, or in relation to contempt of court, defamation, or incitement to an offence."),
    ("21", "Protection of life and personal liberty", "constitutional_law",
     "Article 21 — No person shall be deprived of his life or personal liberty except according to procedure established by law. This has been interpreted broadly to include the right to live with dignity, right to livelihood, right to shelter, right to clean environment, right to health, right to education, right to privacy, and right to speedy trial. It is the most expansive fundamental right."),
    ("21A", "Right to education", "constitutional_law",
     "Article 21A — The State shall provide free and compulsory education to all children of the age of six to fourteen years in such manner as the State may, by law, determine. Inserted by the Constitution (Eighty-sixth Amendment) Act, 2002. Implemented through the Right of Children to Free and Compulsory Education Act, 2009 (RTE Act)."),
    ("22", "Protection against arrest and detention", "constitutional_law",
     "Article 22 — (1) No person who is arrested shall be detained in custody without being informed, as soon as may be, of the grounds for such arrest. (2) Every person arrested and detained shall be produced before the nearest magistrate within twenty-four hours (excluding travel time). (3) No such person shall be detained beyond said period without the authority of a magistrate. Every arrested person has the right to consult and be defended by a legal practitioner of their choice."),
    ("32", "Remedies for enforcement of fundamental rights", "constitutional_law",
     "Article 32 — The right to move the Supreme Court by appropriate proceedings for the enforcement of fundamental rights is guaranteed. The Supreme Court shall have power to issue directions or orders or writs (habeas corpus, mandamus, prohibition, quo warranto, certiorari) for enforcement of any of the rights conferred by Part III. This is itself a fundamental right — Dr. B.R. Ambedkar called it 'the heart and soul of the Constitution'."),
    ("226", "Power of High Courts to issue writs", "constitutional_law",
     "Article 226 — Every High Court shall have the power to issue directions, orders or writs (habeas corpus, mandamus, prohibition, quo warranto, certiorari) for the enforcement of fundamental rights and for any other purpose. Unlike Article 32 which is limited to fundamental rights, Article 226 can be used for enforcement of any legal right. A writ petition under Article 226 is filed in the High Court of the state."),
    ("300A", "Right to property", "constitutional_law",
     "Article 300A — No person shall be deprived of his property save by authority of law. While no longer a fundamental right (removed by 44th Amendment, 1978), it remains a constitutional right. Any acquisition of property by the State must be backed by law and must be for a public purpose. Compensation must be paid."),
    ("311", "Dismissal of civil servants", "constitutional_law",
     "Article 311 — No person who is a member of a civil service shall be dismissed or removed by an authority subordinate to that by which they were appointed. No such person shall be dismissed or removed or reduced in rank except after an inquiry with reasonable opportunity of being heard. Government employees have protection against arbitrary dismissal."),
    ("15", "Prohibition of discrimination", "constitutional_law",
     "Article 15 — The State shall not discriminate against any citizen on grounds only of religion, race, caste, sex, place of birth or any of them. No citizen shall be subject to any disability, liability, restriction or condition on these grounds with regard to access to shops, public restaurants, hotels, wells, tanks, bathing ghats, roads and places of public resort."),
    ("23", "Prohibition of traffic in human beings and forced labour", "constitutional_law",
     "Article 23 — Traffic in human beings and begar (forced labour) and other similar forms of forced labour are prohibited and any contravention shall be an offence punishable in accordance with law. This article protects against bonded labour, human trafficking, and any form of forced labour."),
    ("25", "Freedom of conscience and free profession, practice and propagation of religion", "constitutional_law",
     "Article 25 — Subject to public order, morality and health, all persons are equally entitled to freedom of conscience and the right freely to profess, practise and propagate religion. The State may regulate or restrict any economic, financial, political or other secular activity associated with religious practice."),
    # Preamble
    ("Preamble", "Preamble to the Constitution of India", "constitutional_law",
     "WE, THE PEOPLE OF INDIA, having solemnly resolved to constitute India into a SOVEREIGN SOCIALIST SECULAR DEMOCRATIC REPUBLIC and to secure to all its citizens: JUSTICE, social, economic and political; LIBERTY of thought, expression, belief, faith and worship; EQUALITY of status and of opportunity; and to promote among them all FRATERNITY assuring the dignity of the individual and the unity and integrity of the Nation. The Preamble declares India a sovereign, socialist, secular, democratic republic and sets out the objectives of justice, liberty, equality and fraternity."),
    # DPSPs relevant to legal triage
    ("39A", "Equal justice and free legal aid", "constitutional_law",
     "Article 39A — The State shall secure that the operation of the legal system promotes justice, on a basis of equal opportunity, and shall, in particular, provide free legal aid, by suitable legislation or schemes, to ensure that opportunities for securing justice are not denied to any citizen by reason of economic or other disabilities. This is the constitutional basis for Legal Services Authorities and free legal aid."),
    ("41", "Right to work, education and public assistance", "constitutional_law",
     "Article 41 — The State shall, within the limits of its economic capacity, make effective provision for securing the right to work, to education and to public assistance in cases of unemployment, old age, sickness and disablement."),
]

key_article_nums = set()
for num, title, doc_type, text in KEY_ARTICLES:
    chunk_id = f'CONST_A{num}'
    # Don't duplicate if already extracted from PDF
    if chunk_id not in {c['chunk_id'] for c in constitution_chunks}:
        constitution_chunks.append({
            'chunk_id': chunk_id,
            'source': 'Constitution_of_India',
            'doc_type': doc_type,
            'title': f'Article {num}: {title}',
            'text': text,
        })
        key_article_nums.add(num)

print(f'  📜 Total Constitution chunks: {len(constitution_chunks)} ({len(key_article_nums)} curated key articles added)')

In [0]:
supplementary_chunks = []

# ── Consumer Protection Act 2019 ────────────────────────────────────────────
CPA_CHUNKS = [
    ("CPA_S2", "Section 2 — Definitions", "consumer_law",
     "Consumer Protection Act 2019, Section 2 — Key definitions: 'Consumer' means any person who buys goods or hires services for consideration (includes online/electronic transactions). 'Defect' means any fault, imperfection or shortcoming in quality, quantity, potency, purity or standard. 'Deficiency' means any fault, imperfection, shortcoming or inadequacy in the quality, nature and manner of performance of a service. 'Unfair trade practice' includes false representation, misleading advertisement, refusing to withdraw defective goods."),
    ("CPA_S35", "Section 35 — Jurisdiction of District Commission", "consumer_law",
     "Consumer Protection Act 2019, Section 35 — The District Consumer Disputes Redressal Commission (District Commission) has jurisdiction over complaints where the value of goods or services paid as consideration does not exceed Rs. 1 crore. Complaints must be filed in the district where the opposite party resides or carries on business, or where the cause of action arose. Filing fee: ₹100 for claims up to ₹5 lakhs; ₹200 for ₹5-10 lakhs; ₹400 for ₹10-20 lakhs; ₹500 for ₹20-50 lakhs; ₹2000 for ₹50 lakhs to ₹1 crore."),
    ("CPA_S38", "Section 38 — Manner of filing complaint", "consumer_law",
     "Consumer Protection Act 2019, Section 38 — A complaint may be filed by: (a) the consumer; (b) any recognised consumer association; (c) the Central Government or State Government; (d) one or more consumers with same interest (class action); (e) legal representative in case of death. Complaint must contain: name and address of complainant and opposite party, facts of complaint, relief sought, and documents supporting the claim. Can be filed online through edaakhil.nic.in portal."),
    ("CPA_S39", "Section 39 — Admissibility of complaints", "consumer_law",
     "Consumer Protection Act 2019, Section 39 — Time limit: Complaint must be filed within 2 years from the date on which the cause of action arose. The Commission may condone the delay if the complainant shows sufficient cause. Complaints relating to goods: defect in goods, unfair trade practices, hazardous goods. Complaints relating to services: deficiency in service, charging excess price, offering hazardous services."),
    ("CPA_S69", "Section 69 — Product liability", "consumer_law",
     "Consumer Protection Act 2019, Section 69 — Product Liability: A product liability action may be brought by a complainant against a product manufacturer, product service provider, or product seller for any harm caused by a defective product. The manufacturer is liable if the product contains a manufacturing defect, is defective in design, or there is a deviation from manufacturing specifications. The product seller is liable if they have exercised substantial control over the design, testing, manufacture, packaging, or labelling."),
    ("CPA_CCPA", "Central Consumer Protection Authority (CCPA)", "consumer_law",
     "Consumer Protection Act 2019 — CCPA (Central Consumer Protection Authority) established under Section 10 to regulate matters relating to violation of consumer rights, unfair trade practices, and false or misleading advertisements. CCPA can: (a) conduct inquiry/investigation; (b) recall/withdraw unsafe goods; (c) pass orders to reimburse/return goods; (d) impose penalty for misleading ads up to ₹10 lakhs (₹50 lakhs for repeat offence). Consumer Helpline: 1800-11-4000 (toll-free)."),
]

# ── Protection of Women from Domestic Violence Act 2005 ─────────────────────
DV_CHUNKS = [
    ("DV_S3", "Section 3 — Definition of domestic violence", "family_law",
     "Protection of Women from Domestic Violence Act 2005, Section 3 — Domestic violence includes: (a) Physical abuse — any act causing bodily pain, harm, danger to life, limb, health; (b) Sexual abuse — any conduct of sexual nature that abuses, humiliates, degrades; (c) Verbal and emotional abuse — insults, ridicule, name-calling, threats to cause physical pain; (d) Economic abuse — deprivation of financial resources, disposal of household effects, prohibition from accessing resources. Threats of any of these also constitute domestic violence."),
    ("DV_S4", "Section 4 — Information to Protection Officer", "family_law",
     "DV Act 2005, Section 4 — Any person who has reason to believe that an act of domestic violence has been committed may give information about it to a Protection Officer. Every state has Protection Officers appointed by the State Government. The Protection Officer shall make a Domestic Incident Report (DIR) and forward copies to the Magistrate and the service providers in the area."),
    ("DV_S12", "Section 12 — Application to Magistrate", "family_law",
     "DV Act 2005, Section 12 — An aggrieved person or a Protection Officer or any other person on behalf of the aggrieved person may present an application to the Magistrate seeking one or more reliefs under this Act. The Magistrate shall fix the first hearing within 3 days of receiving the application and shall endeavour to dispose of every application within 60 days. Filing is FREE — no court fees required."),
    ("DV_S17", "Section 17 — Right to reside in shared household", "family_law",
     "DV Act 2005, Section 17 — Every woman in a domestic relationship has the right to reside in the shared household, whether or not she has any title or rights in the shared household. The aggrieved person shall not be evicted or excluded from the shared household by the respondent. The Magistrate may restrain the respondent from dispossessing or disturbing the possession of the aggrieved person."),
    ("DV_S18_19", "Sections 18-19 — Protection and residence orders", "family_law",
     "DV Act 2005, Section 18 (Protection Orders) — The Magistrate may prohibit the respondent from: committing any act of domestic violence; aiding or abetting domestic violence; entering the aggrieved person's place of employment; attempting to communicate with the aggrieved person. Section 19 (Residence Orders) — may restrain respondent from dispossessing the aggrieved person; or require respondent to arrange alternate accommodation."),
    ("DV_HELPLINES", "Domestic Violence helplines and resources", "family_law",
     "Domestic Violence emergency helplines: Women Helpline: 181 (24x7, toll-free, all states). Police Emergency: 112. National Commission for Women (NCW): 7827-170-170. One Stop Centre (Sakhi): Available in every district — provides medical aid, police assistance, legal aid, psycho-social counselling and temporary shelter. For Protection Officer, contact the District Women & Child Development office. Online complaint: ncw.nic.in. Free legal aid available through District Legal Services Authority (DLSA) — call NALSA helpline 15100."),
]

# ── Right to Information Act 2005 ───────────────────────────────────────────
RTI_CHUNKS = [
    ("RTI_S6", "Section 6 — Request for obtaining information", "constitutional_law",
     "Right to Information Act 2005, Section 6 — A person who desires to obtain any information shall make a request in writing or through electronic means to the Public Information Officer (PIO) of the concerned public authority. Specify the particulars of information sought. NOT required to give reasons for requesting information. Fee: ₹10 (for Central Government departments; states may vary). BPL applicants are exempt from fees. Application can be in English, Hindi, or the official language of the area."),
    ("RTI_S7", "Section 7 — Disposal of request", "constitutional_law",
     "RTI Act 2005, Section 7 — The PIO shall provide information within 30 days of receipt of the request. If the information concerns the life or liberty of a person, it shall be provided within 48 hours. If the PIO fails to give a decision within the specified period, the request shall be deemed to have been refused. Additional fees may be charged for providing information (photocopying at ₹2/page for A4 size)."),
    ("RTI_S19", "Section 19 — Appeal", "constitutional_law",
     "RTI Act 2005, Section 19 — First Appeal: If no response within 30 days, or if dissatisfied with the response, file first appeal with the First Appellate Authority (FAA, officer senior to PIO) within 30 days. Second Appeal: If dissatisfied with FAA's decision, file second appeal with the Central/State Information Commission within 90 days. The Information Commission can impose a penalty of ₹250 per day on the PIO (max ₹25,000) for failure to provide information without reasonable cause."),
    ("RTI_HOWTO", "How to file an RTI application — step by step", "constitutional_law",
     "How to file RTI: Step 1 — Identify the public authority that holds the information. Step 2 — Write application addressed to PIO of that authority, stating 'Under Section 6(1) of RTI Act 2005...'. Step 3 — Pay fee of ₹10 (by cash/DD/IPO/online). Step 4 — Submit by post, in person, or online at rtionline.gov.in (for Central Government). Step 5 — Get acknowledgment with date. Step 6 — Wait 30 days for response. Step 7 — If no response or inadequate response, file First Appeal within 30 days to FAA. Online portal: rtionline.gov.in (Central), state RTI portals for state departments."),
]

# ── Labour Laws ─────────────────────────────────────────────────────────────
LABOUR_CHUNKS = [
    ("LAB_ID_TERMIN", "Wrongful termination under Industrial Disputes Act", "labour_law",
     "Industrial Disputes Act 1947 — Section 25F: No workman who has been in continuous service for not less than one year shall be retrenched unless: (a) one month's notice in writing with reasons, or wages in lieu of notice; (b) compensation equal to 15 days' average pay for every completed year of continuous service; (c) notice served on the appropriate Government. Section 25N: In establishments with 100+ workmen, prior permission of the appropriate Government is required for retrenchment. Termination without following this procedure is illegal and the workman is entitled to reinstatement with back wages."),
    ("LAB_GRATUITY", "Payment of Gratuity Act 1972", "labour_law",
     "Payment of Gratuity Act 1972 — Applies to factories, mines, oilfields, plantations, ports, railways, establishments with 10+ employees. Eligibility: Employee who has completed 5 years of continuous service (relaxed in case of death/disablement). Amount: 15 days' wages × years of service (based on last drawn wages). Maximum limit: ₹20 lakhs. Filing: Application to employer within 30 days of eligibility. If employer fails to pay, file complaint to the Controlling Authority (Labour Commissioner). Helpline: Shram Suvidha 1800-11-6763 (toll-free)."),
    ("LAB_WAGES", "Payment of Wages Act 1936 / Minimum Wages Act", "labour_law",
     "Payment of Wages Act 1936 — Wages must be paid before the 7th day of the following month (for establishments with <1000 workers) or 10th day (for 1000+ workers). No unauthorized deductions. If wages are delayed or deducted illegally, file complaint with the Labour Commissioner within 12 months. Minimum Wages Act 1948 — State-specific minimum wages apply. Central minimum wage for scheduled employment: varies by state and occupation. Check latest rates at labour.gov.in."),
    ("LAB_POSH", "Prevention of Sexual Harassment at Workplace (POSH Act 2013)", "labour_law",
     "Sexual Harassment of Women at Workplace (Prevention, Prohibition and Redressal) Act 2013 — Every employer with 10+ employees must constitute an Internal Complaints Committee (ICC). Complaint must be filed within 3 months of the incident (extendable by 3 months). ICC must complete inquiry within 90 days. If employer fails to constitute ICC, penalty up to ₹50,000. If employer fails to act on ICC recommendations, penalty up to ₹50,000 (doubled for repeat offence) and cancellation of licence. SHe-Box (online complaint): shebox.nic.in. Women Helpline: 181."),
    ("LAB_EPF", "Employees' Provident Fund (EPF)", "labour_law",
     "Employees' Provident Funds and Miscellaneous Provisions Act 1952 — Applies to establishments with 20+ employees. Employee contribution: 12% of basic wages. Employer contribution: 12% of basic wages (3.67% to EPF, 8.33% to EPS). Total balance can be checked on EPFO portal: unifiedportal-mem.epfindia.gov.in. Online PF withdrawal: Submit Form 19 online through UAN portal. Helpline: EPFO 1800-118-005 (toll-free). If employer is not depositing PF, file complaint on epfigms.gov.in."),
]

# ── Property Law ────────────────────────────────────────────────────────────
PROPERTY_CHUNKS = [
    ("PROP_TPA", "Transfer of Property Act 1882 — tenant rights", "property_law",
     "Transfer of Property Act 1882, Section 106 — Lease of immovable property: A lease of immovable property for agricultural or manufacturing purposes shall be deemed to be a lease from year to year, terminable by six months' notice. For other purposes, from month to month, terminable by fifteen days' notice. Notice must expire with the end of a month of the tenancy. Section 108: Rights of lessee include right to possession, right to use the property, right to improvements, right to assignment/subletting (unless prohibited by lease)."),
    ("PROP_RERA", "Real Estate (Regulation and Development) Act 2016 — RERA", "property_law",
     "RERA 2016 — All real estate projects with land over 500 sq. metres or 8+ apartments must be registered with the state RERA authority. Buyer rights: (a) obtain information about project; (b) claim possession on time; (c) claim refund with interest if project delayed; (d) structural defect liability for 5 years. Complaint to RERA authority within 1 year. Appeal to RERA Appellate Tribunal within 60 days. RERA portal: rera.gov.in (central), state-specific RERA portals."),
    ("PROP_ENCROACH", "Property encroachment and legal remedies", "property_law",
     "Property encroachment remedies in India: (1) Send legal notice to the encroacher demanding removal within 15-30 days. (2) File a civil suit for permanent injunction and possession in the Civil Court. (3) File a complaint before the Revenue/Tahsildar office for encroachment on government or private land. (4) If recent encroachment (within 6 months), file suit under Specific Relief Act Section 6 for recovery of possession. (5) For urgent relief, apply for temporary injunction under Order 39 CPC. Court fee: varies by state (ad valorem on property value). Keep property documents, survey records, tax receipts as evidence."),
]

# Combine all supplementary chunks
for label, chunks_list in [("CPA", CPA_CHUNKS), ("DV", DV_CHUNKS), ("RTI", RTI_CHUNKS), ("Labour", LABOUR_CHUNKS), ("Property", PROPERTY_CHUNKS)]:
    for chunk_id, title, doc_type, text in chunks_list:
        supplementary_chunks.append({
            'chunk_id': chunk_id,
            'source': label,
            'doc_type': doc_type,
            'title': title,
            'text': text,
        })
    print(f'  ✅ {label}: {len(chunks_list)} chunks')

print(f'\n📚 Total supplementary chunks: {len(supplementary_chunks)}')

In [0]:
# Combine Constitution + supplementary
new_chunks = constitution_chunks + supplementary_chunks
df_new = pd.DataFrame(new_chunks)

print(f'📊 New chunks to add: {len(df_new)}')
print(df_new.groupby('source')['chunk_id'].count())

# Check if corpus table exists and current count
try:
    existing = spark.table(f'{CATALOG}.{SCHEMA}.legal_rag_corpus').count()
    print(f'\n📊 Existing corpus: {existing} chunks')
except Exception:
    existing = 0
    print('\n⚠️  No existing corpus table — will create new one')

# Remove any existing Constitution/supplementary chunks to avoid duplicates on re-run
if existing > 0:
    from pyspark.sql import functions as F
    new_sources = df_new['source'].unique().tolist()
    print(f'   Removing old chunks from sources: {new_sources}')
    
    existing_df = spark.table(f'{CATALOG}.{SCHEMA}.legal_rag_corpus')
    filtered_df = existing_df.filter(~F.col('source').isin(new_sources))
    
    # Write filtered existing + new
    new_sdf = spark.createDataFrame(df_new.astype(str))
    combined = filtered_df.unionByName(new_sdf, allowMissingColumns=True)
    combined.write.format('delta').mode('overwrite') \
        .option('overwriteSchema', 'true') \
        .saveAsTable(f'{CATALOG}.{SCHEMA}.legal_rag_corpus')
    final_count = spark.table(f'{CATALOG}.{SCHEMA}.legal_rag_corpus').count()
    print(f'\n✅ legal_rag_corpus updated: {final_count} total chunks')
else:
    # No existing table — just save new chunks
    save_table(df_new, 'legal_rag_corpus', mode='overwrite')
    print(f'\n✅ legal_rag_corpus created: {len(df_new)} chunks')

In [0]:
display(
    spark.sql(f"""
        SELECT source, doc_type, COUNT(*) as chunks
        FROM {CATALOG}.{SCHEMA}.legal_rag_corpus
        GROUP BY source, doc_type
        ORDER BY chunks DESC
    """)
)
